# Corporate Signal Intelligence
## Caderno 01: Coleta de Dados

Duas fontes públicas alimentam todo o projeto: a base **Stooq**, para preços diários, e a
base **SEC EDGAR**, para divulgações corporativas e fundamentos em formato XBRL. Nenhuma das
duas é difícil de consultar. Ambas, porém, são fáceis de consultar **incorretamente**, de
maneira a produzir um conjunto de dados aparentemente completo mas com anos de histórico
silenciosamente ausentes. Este caderno documenta as duas armadilhas porque a versão anterior
do pipeline caiu nas duas.

| Camada | Fonte | Granularidade | Arquivo produzido |
|---|---|---|---|
| Mercado | Serviço CSV da Stooq | ticker × pregão | `stooq_market_raw.csv` |
| Divulgação | API de submissões da EDGAR | ticker × formulário | `sec_filings_full_raw.csv` |
| Fundamentos | Fatos XBRL da EDGAR | ticker × conceito × período | `sec_company_facts_full_raw.csv` |

### O universo analisado

Dez companhias de tecnologia de grande capitalização listadas nos Estados Unidos: AAPL, MSFT,
NVDA, GOOGL, AMZN, META, TSLA, AMD, INTC e ORCL. A homogeneidade é deliberada: um único setor
e um único calendário de negociação garantem que um dia sinalizado como atípico o seja em
relação a um grupo comparável, e não em relação a uma mistura artificial de modelos de negócio.

### O que este caderno estabelece

1. Por que a fonte de dados de mercado deixou de ser a Alpha Vantage e passou a ser a Stooq.
2. **Armadilha de truncamento nº 1:** o serviço de submissões devolve apenas os formulários
   mais recentes no bloco principal e esconde o restante atrás de paginação.
3. **Armadilha de truncamento nº 2:** um ticker não é uma entidade jurídica, e coletar pelo CIK
   vigente elimina o histórico de qualquer companhia que tenha passado por reorganização
   societária.
4. A coleta corrigida e a validação de que ela capturou o que deveria capturar.

In [1]:
# Dependências da etapa de coleta

%pip install -q pandas requests python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Ambiente e configuração

import os
import sys
import time
import warnings
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

sys.path.append("scripts")
warnings.filterwarnings("ignore")
load_dotenv()

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 170)

DATA_DIR = Path("data")

TICKERS = [
    "AAPL", "MSFT", "NVDA", "GOOGL", "AMZN",
    "META", "TSLA", "AMD", "INTC", "ORCL",
]

STOOQ_API_KEY = os.getenv("STOOQ_API_KEY")

# A EDGAR exige um User-Agent que identifique quem faz a chamada; não exige chave de acesso.
SEC_USER_AGENT = os.getenv(
    "SEC_USER_AGENT",
    "Corporate Signal Intelligence (MBA USP/Esalq research) addoqyn@gmail.com",
)

print("Chave da Stooq presente:", bool(STOOQ_API_KEY))
print("User-Agent da SEC:", SEC_USER_AGENT)

Chave da Stooq presente: False
User-Agent da SEC: Corporate Signal Intelligence (MBA USP/Esalq research) addoqyn@gmail.com


---

## 1. Dados de mercado

### Por que não a Alpha Vantage

A primeira implementação usava a Alpha Vantage. Ela funciona, mas é inviável para este
projeto. O limite diário de requisições do plano gratuito transforma a coleta de dez
históricos completos em um exercício de vários dias, e o endpoint de séries ajustadas está
restrito ao plano pago. Em vez de construir o pipeline em torno de uma cota, trocou-se a
fonte.

### Stooq

A Stooq expõe um endpoint CSV simples que devolve, em uma única requisição, o histórico
completo já ajustado por proventos. O caderno de limpeza verifica essa afirmação em vez de
confiar nela: os maiores movimentos diários da série resultante recaem sobre eventos de
mercado documentados, a Segunda-Feira Negra e o alerta de resultados da Apple em setembro de
2000, por exemplo, e não sobre datas conhecidas de desdobramento, que é exatamente a
assinatura esperada de uma série ajustada.

In [3]:
# Baixando o histórico diário completo de um símbolo

def fetch_stooq_daily(ticker: str) -> pd.DataFrame:
    """Baixa o histórico diário completo, já ajustado, de um único símbolo."""
    if not STOOQ_API_KEY:
        raise RuntimeError("STOOQ_API_KEY não está definida")

    response = requests.get(
        "https://stooq.com/q/d/l/",
        params={"s": f"{ticker.lower()}.us", "i": "d", "apikey": STOOQ_API_KEY},
        headers={"User-Agent": "corporate-signal-intelligence/1.0"},
        timeout=30,
    )
    response.raise_for_status()
    payload = response.text.strip()

    # Sem chave válida o endpoint responde com uma página de desafio em JavaScript, e não
    # com um erro HTTP: por isso a resposta precisa ser inspecionada antes do parsing.
    if not payload.startswith("Date,Open,High,Low,Close,Volume"):
        raise RuntimeError(f"A Stooq não devolveu CSV para {ticker}: {payload[:80]}")

    frame = pd.read_csv(StringIO(payload)).rename(columns={
        "Date": "date", "Open": "open", "High": "high",
        "Low": "low", "Close": "close", "Volume": "volume",
    })
    frame["ticker"] = ticker.upper()
    frame["date"] = pd.to_datetime(frame["date"])
    frame["source"] = "stooq"
    frame["collected_at"] = datetime.now(timezone.utc)

    return frame[[
        "ticker", "date", "open", "high", "low", "close", "volume",
        "source", "collected_at",
    ]]


def collect_market_data(tickers: list[str]) -> pd.DataFrame:
    frames = []
    for ticker in tickers:
        frame = fetch_stooq_daily(ticker)
        frames.append(frame)
        print(f"  {ticker}: {len(frame):>6} linhas | "
              f"{frame['date'].min():%Y-%m-%d} a {frame['date'].max():%Y-%m-%d}")
        time.sleep(0.5)
    return pd.concat(frames, ignore_index=True).sort_values(["ticker", "date"])


market_path = DATA_DIR / "stooq_market_raw.csv"

if STOOQ_API_KEY:
    market_raw_df = collect_market_data(TICKERS)
    market_raw_df.to_csv(market_path, index=False)
    print(f"\nGravado em {market_path}")
else:
    # A chave não está disponível neste ambiente, então o arquivo coletado anteriormente é
    # reaproveitado. A janela do estudo encerra-se na última data dele; ver caderno 02.
    market_raw_df = pd.read_csv(market_path, parse_dates=["date"])
    print("STOOQ_API_KEY não definida: reaproveitando o arquivo coletado anteriormente.")

market_raw_df.groupby("ticker").agg(
    linhas=("date", "size"), inicio=("date", "min"), fim=("date", "max"),
).reindex(TICKERS)

STOOQ_API_KEY não definida: reaproveitando o arquivo coletado anteriormente.


,linhas,inicio,fim
ticker,,,
AAPL,10507,1984-09-07,2026-05-21
MSFT,10124,1986-03-13,2026-05-21
NVDA,6874,1999-01-22,2026-05-21
GOOGL,5474,2004-08-19,2026-05-21
AMZN,7294,1997-05-16,2026-05-21
META,3522,2012-05-18,2026-05-21
TSLA,3999,2010-06-28,2026-05-21
AMD,10878,1983-03-21,2026-05-21
INTC,13697,1972-01-07,2026-05-21


---

## 2. SEC EDGAR: o ponto de entrada

A EDGAR indexa tudo pelo **CIK**, o *Central Index Key* da entidade que protocola o documento.
O mapeamento entre ticker e CIK é publicado como um único arquivo JSON, e aponta sempre para a
entidade que protocola **hoje**.

É justamente esse último detalhe que constitui o problema, demonstrado na Seção 3.

In [4]:
# O mapeamento público entre ticker e CIK

def sec_get_json(url: str) -> dict:
    response = requests.get(url, headers={"User-Agent": SEC_USER_AGENT}, timeout=60)
    response.raise_for_status()
    time.sleep(0.4)  # a SEC pede menos de 10 requisições por segundo
    return response.json()


ticker_map = sec_get_json("https://www.sec.gov/files/company_tickers.json")
companies_df = pd.DataFrame([
    {
        "ticker": entry["ticker"],
        "cik": str(entry["cik_str"]).zfill(10),
        "company_name": entry["title"],
        "source": "sec_edgar",
        "collected_at": datetime.now(timezone.utc),
    }
    for entry in ticker_map.values()
])

selected_companies_df = (
    companies_df[companies_df["ticker"].isin(TICKERS)].sort_values("ticker").reset_index(drop=True)
)
selected_companies_df.to_csv(DATA_DIR / "sec_companies_selected.csv", index=False)

selected_companies_df

,ticker,cik,company_name,source,collected_at
0,AAPL,0000320193,Apple Inc.,sec_edgar,2026-08-07 09:03:36.857479+00:00
1,AMD,0000002488,ADVANCED MICRO DEVICES INC,sec_edgar,2026-08-07 09:03:36.862410+00:00
2,AMZN,0001018724,AMAZON COM INC,sec_edgar,2026-08-07 09:03:36.857482+00:00
3,GOOGL,0001652044,Alphabet Inc.,sec_edgar,2026-08-07 09:03:36.857480+00:00
4,INTC,0000050863,INTEL CORP,sec_edgar,2026-08-07 09:03:36.857488+00:00
5,META,0001326801,"Meta Platforms, Inc.",sec_edgar,2026-08-07 09:03:36.857483+00:00
6,MSFT,0000789019,MICROSOFT CORP,sec_edgar,2026-08-07 09:03:36.857481+00:00
7,NVDA,0001045810,NVIDIA CORP,sec_edgar,2026-08-07 09:03:36.857475+00:00
8,ORCL,0001341439,ORACLE CORP,sec_edgar,2026-08-07 09:03:36.857500+00:00
9,TSLA,0001318605,"Tesla, Inc.",sec_edgar,2026-08-07 09:03:36.857484+00:00


---

## 3. Duas maneiras de perder uma década de histórico

As duas armadilhas produzem um conjunto de dados que parece completo. Nenhuma delas gera erro.
A única forma de percebê-las é verificar o que a resposta de fato contém, que é o que esta
seção faz.

### Armadilha nº 1: o serviço de submissões é paginado

O endpoint `https://data.sec.gov/submissions/CIK##########.json` devolve um bloco
`filings.recent` e, para companhias que protocolam com frequência, uma lista `filings.files`
apontando para documentos adicionais que guardam todo o restante. Um coletor que leia apenas
`filings.recent` obtém cerca dos mil formulários mais recentes e descarta silenciosamente o
resto.

In [5]:
# Quanto de histórico vive fora do bloco filings.recent?

pagination_rows = []
for ticker, cik in [("AAPL", 320193), ("MSFT", 789019), ("INTC", 50863)]:
    submissions = sec_get_json(f"https://data.sec.gov/submissions/CIK{cik:010d}.json")
    recent = submissions["filings"]["recent"]
    archived = submissions["filings"].get("files", [])

    pagination_rows.append({
        "ticker": ticker,
        "no_bloco_recente": len(recent["accessionNumber"]),
        "bloco_comeca_em": min(recent["filingDate"]),
        "documentos_de_arquivo": len(archived),
        "formularios_no_arquivo": sum(item["filingCount"] for item in archived),
        "inicio_verdadeiro": min(
            [min(recent["filingDate"])] + [item["filingFrom"] for item in archived]
        ),
    })

pagination_df = pd.DataFrame(pagination_rows).set_index("ticker")
pagination_df["anos_perdidos_sem_paginacao"] = (
    (pd.to_datetime(pagination_df["bloco_comeca_em"])
     - pd.to_datetime(pagination_df["inicio_verdadeiro"])).dt.days / 365.25
).round(1)

pagination_df

,no_bloco_recente,bloco_comeca_em,documentos_de_arquivo,formularios_no_arquivo,inicio_verdadeiro,anos_perdidos_sem_paginacao
ticker,,,,,,
AAPL,1000,2015-06-01,1,1238,1994-01-26,21.3
MSFT,1000,2020-04-30,2,3481,1994-02-14,26.2
INTC,1002,2019-02-27,2,2965,1994-02-10,25.0


Para essas companhias o bloco principal alcança apenas os últimos anos. Tudo o que é anterior
está nos documentos de arquivo, e ler somente `filings.recent` teria custado de duas a três
décadas de histórico de divulgação por empresa.

### Armadilha nº 2: um ticker não é uma entidade jurídica

A Alphabet Inc. foi criada em 2015. A Google Inc. protocolou tudo o que veio antes disso sob
um CIK diferente, e a EDGAR indexa os formulários pela entidade, não pelo ticker. O mesmo vale
para a Oracle, reorganizada em 2005.

In [6]:
# O que o CIK vigente conhece, e o que ele desconhece

lineage_probe = []
for label, cik in [
    ("GOOGL (Alphabet Inc. (vigente)", 1652044),
    ("GOOGL) Google Inc. (predecessora)", 1288776),
    ("ORCL, Oracle Corporation (vigente)", 1341439),
    ("ORCL, Oracle Systems Corp. (predecessora)", 777676),
]:
    submissions = sec_get_json(f"https://data.sec.gov/submissions/CIK{cik:010d}.json")
    recent = submissions["filings"]["recent"]
    archived = submissions["filings"].get("files", [])
    earliest = min([min(recent["filingDate"])] + [item["filingFrom"] for item in archived])

    lineage_probe.append({
        "entidade": label,
        "cik": cik,
        "nome_no_registro": submissions["name"],
        "primeiro_formulario": earliest,
        "ultimo_formulario": max(recent["filingDate"]),
    })

pd.DataFrame(lineage_probe).set_index("entidade")

,cik,nome_no_registro,primeiro_formulario,ultimo_formulario
entidade,,,,
GOOGL (Alphabet Inc. (vigente),1652044,Alphabet Inc.,2015-10-02,2026-08-06
GOOGL) Google Inc. (predecessora),1288776,GOOGLE INC.,2001-03-12,2026-07-14
"ORCL, Oracle Corporation (vigente)",1341439,ORACLE CORP,2005-10-19,2026-07-28
"ORCL, Oracle Systems Corp. (predecessora)",777676,Oracle Systems,1994-01-11,2006-08-10


A sondagem torna a perda concreta. O registro próprio da Alphabet começa em outubro de 2015,
de modo que uma coleta indexada pelo CIK vigente inicia o histórico da GOOGL onze anos após a
abertura de capital da companhia: a oferta inicial, todo o período pós-crise e a transição
para o mobile ficam de fora. A entidade atual da Oracle começa em 2005, descartando uma
década.

Há uma complicação que uma união ingênua trataria de forma errada: **a entidade predecessora
continua protocolando documentos depois que a sucessora aparece.** A Google Inc. segue
apresentando formulários como subsidiária da Alphabet, em sua maioria da Seção 16, relativos a
negociações de pessoas ligadas à companhia. Simplesmente concatenar as duas entidades
duplicaria os períodos sobrepostos.

A correção consiste em tratar o ticker como uma *sequência ordenada* de entidades, atribuindo
a cada predecessora uma janela de validade que se encerra no primeiro formulário da sucessora,
e então deduplicar pelo número de protocolo. É o que o módulo
`scripts/collect_sec_full_history.py` implementa, e é esse módulo que este caderno invoca em
vez de repetir a lógica aqui.

---

## 4. A coleta corrigida

O coletor percorre todas as entidades de todos os tickers, segue a paginação, aplica a janela
de validade de cada predecessora e, na mesma passagem, extrai os conceitos XBRL selecionados.
São nove conceitos: receita sob as duas etiquetas: anterior e posterior à norma ASC 606,
lucro líquido, resultado operacional, despesa de P&D, ativo total, passivo total, patrimônio
líquido e caixa.

In [7]:
# Executando o coletor com resolução de linhagem societária

from collect_sec_full_history import ENTITIES, TARGET_CONCEPTS, collect

print("Mapa de entidades:")
for ticker, entities in ENTITIES.items():
    for entity in entities:
        window = f" até {entity['until']}" if entity.get("until") else ""
        print(f"  {ticker:<6} CIK {entity['cik']:010d}  {entity['name']}{window}")

print(f"\nConceitos XBRL requisitados: {len(TARGET_CONCEPTS)}")

Mapa de entidades:
  AAPL   CIK 0000320193  Apple Inc.
  MSFT   CIK 0000789019  Microsoft Corporation
  NVDA   CIK 0001045810  NVIDIA Corporation
  AMZN   CIK 0001018724  Amazon.com, Inc.
  META   CIK 0001326801  Meta Platforms, Inc.
  TSLA   CIK 0001318605  Tesla, Inc.
  AMD    CIK 0000002488  Advanced Micro Devices, Inc.
  INTC   CIK 0000050863  Intel Corporation
  GOOGL  CIK 0001652044  Alphabet Inc.
  GOOGL  CIK 0001288776  Google Inc. até 2015-10-02
  ORCL   CIK 0001341439  Oracle Corporation
  ORCL   CIK 0000777676  Oracle Systems Corporation até 2005-10-19

Conceitos XBRL requisitados: 9


In [8]:
# Coletando formulários e fundamentos

filings_raw_df, facts_raw_df = collect()

filings_raw_df.to_csv(DATA_DIR / "sec_filings_full_raw.csv", index=False)
facts_raw_df.to_csv(DATA_DIR / "sec_company_facts_full_raw.csv", index=False)

print(f"\nformulários: {filings_raw_df.shape}")
print(f"fundamentos: {facts_raw_df.shape}")

AAPL | CIK 0000320193 — Apple Inc.


    filings:   2238  1994-01-26 → 2026-07-31


    facts:     1716
MSFT | CIK 0000789019 — Microsoft Corporation


    filings:   4481  1994-02-14 → 2026-08-06


    facts:     1768
NVDA | CIK 0001045810 — NVIDIA Corporation


...

    filings:   2309  2005-10-19 → 2026-07-28


    facts:     1335
ORCL | CIK 0000777676 — Oracle Systems Corporation (until 2005-10-19)


    filings:    602  1994-01-11 → 2005-10-13


    no XBRL facts for CIK 777676 (Oracle Systems Corporation)



formulários: (37384, 10)
fundamentos: (15495, 17)


---

## 5. A coleta trouxe o que deveria trazer?

Três verificações, porque "a requisição foi bem-sucedida" não é o mesmo que "o dado está lá".

In [9]:
# Verificação 1: cobertura por ticker e alcance histórico obtido

CORE_FORMS = ["10-K", "10-K405", "10-Q", "8-K"]
core_filings = filings_raw_df[filings_raw_df["form_type"].isin(CORE_FORMS)]

coverage_df = core_filings.groupby("ticker").agg(
    formularios_principais=("accession_number", "size"),
    mais_antigo=("filing_date", "min"),
    mais_recente=("filing_date", "max"),
).reindex(TICKERS)

coverage_df["anos_cobertos"] = (
    (coverage_df["mais_recente"] - coverage_df["mais_antigo"]).dt.days / 365.25
).round(1)

coverage_df

,formularios_principais,mais_antigo,mais_recente,anos_cobertos
ticker,,,,
AAPL,363,1994-01-26,2026-07-31,32.5
MSFT,411,1994-02-14,2026-07-29,32.5
NVDA,350,1999-04-29,2026-07-02,27.2
GOOGL,338,2004-07-09,2026-07-23,22.0
AMZN,384,1997-08-14,2026-07-31,29.0
META,188,2012-06-25,2026-07-30,14.1
TSLA,309,2010-08-04,2026-07-23,16.0
AMD,552,1994-01-27,2026-08-05,32.5
INTC,575,1994-03-25,2026-07-24,32.3


In [10]:
# Verificação 2: as duas companhias reorganizadas agora carregam o histórico da predecessora

entity_split = (
    filings_raw_df[filings_raw_df["ticker"].isin(["GOOGL", "ORCL"])].groupby(["ticker", "entity_name"]).agg(formularios=("accession_number", "size"),
         mais_antigo=("filing_date", "min"),
         mais_recente=("filing_date", "max"))
)

print("Nenhum número de protocolo aparece duas vezes:",
      not filings_raw_df.duplicated(["ticker", "accession_number"]).any())
entity_split

Nenhum número de protocolo aparece duas vezes: True


formularios mais_antigo mais_recente
ticker entity_name                                                     
GOOGL  Alphabet Inc.                      2676  2015-10-02   2026-08-06
       Google Inc.                        6370  2001-03-12   2015-10-01
ORCL   Oracle Corporation                 2309  2005-10-19   2026-07-28
       Oracle Systems Corporation          602  1994-01-11   2005-10-13

In [11]:
# Verificação 3: composição por tipo de formulário e o marco inicial do XBRL

form_summary = (
    filings_raw_df["form_type"].value_counts().head(12).rename("formularios").to_frame()
)
form_summary["participacao_pct"] = (
    100 * form_summary["formularios"] / len(filings_raw_df)
).round(1)

print("Primeiro fato XBRL protocolado por emissor (marco da obrigatoriedade de 2009):")
print(facts_raw_df.groupby("ticker")["filing_date"].min().dt.date.to_string())
print()
form_summary

Primeiro fato XBRL protocolado por emissor (marco da obrigatoriedade de 2009):
ticker
AAPL     2009-07-22
AMD      2010-08-04
AMZN     2009-07-24
GOOGL    2009-08-04
INTC     2009-08-03
META     2012-07-31
MSFT     2009-10-23
NVDA     2009-08-20
ORCL     2009-09-21
TSLA     2011-08-12



,formularios,participacao_pct
form_type,,
4,24709,66.1
8-K,2746,7.3
144,1618,4.3
10-Q,818,2.2
SC 13G/A,745,2.0
3,522,1.4
DEFA14A,438,1.2
4/A,367,1.0
PX14A6G,344,0.9


A distribuição por tipo é dominada pelos formulários da Seção 16 (`4`, `3`, `5`) e pelas
comunicações da Regra 144, o que é normal em companhias desse porte e explica por que o
pipeline filtra apenas as três famílias de divulgação que carregam o sinal de interesse. O
marco inicial do XBRL recai em meados de 2009 para todos os emissores, coerente com a
obrigatoriedade escalonada imposta aos grandes declarantes: confirmação útil de que a camada
de fundamentos está completa, e não truncada.

---

## Conclusão

A coleta passa a produzir três conjuntos brutos completos em relação ao que as fontes de fato
disponibilizam:

```text
stooq_market_raw.csv              OHLCV diário, ajustado, histórico integral do provedor
sec_filings_full_raw.csv          todos os formulários, todas as entidades, paginação seguida
sec_company_facts_full_raw.csv    conceitos us-gaap selecionados, a partir da era XBRL
```

Duas lições transcendem este projeto. **Uma API que responde com sucesso não devolveu
necessariamente tudo**: a paginação que esconde o histórico atrás de um índice secundário
passa despercebida justamente porque nada falha. E **um identificador válido hoje pode não
descrever o passado**: qualquer fonte indexada por entidade exige verificação de linhagem
antes que se presuma histórico longo, e ampliar este universo para outra companhia implica
repetir essa checagem.

Nada foi limpo ainda. Padronização de tipos, testes de integridade, normalização de
formulários, tratamento de reapresentações contábeis e o painel de qualidade são tarefa do
próximo caderno.